# Train delay model

Goal: predict the arrival delay (in minutes) at the next stop.

## Step 4. Split the data into train and test

We split **by date**, not randomly. With a random split, neighbouring stops of the same trip
could end up in both parts, and the test result would look better than it really is.

| Part | Dates | Used for |
|---|---|---|
| train | 17-21 Nov | training the final model |
| test | 22-23 Nov | the final check only |

To choose model settings, train is split once more:

| Part | Dates | Used for |
|---|---|---|
| fit | 17-20 Nov | training model variants |
| validation | 21 Nov | comparing the variants |

**Run the cells from top to bottom.**

### 4.1 Connect to Spark

In [1]:
import glob
import os
import sys

# PySpark is installed inside Spark's folder. spark-submit finds it by itself,
# but a Jupyter notebook does not, so we tell Python where to look.
spark_python = os.path.join(os.environ.get("SPARK_HOME", "/opt/spark"), "python")
sys.path[:0] = [spark_python, *glob.glob(f"{spark_python}/lib/py4j-*-src.zip")]

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("RailPredict-Model")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         # Use the same time zone as the data preparation scripts.
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/26 12:03:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 4.2 Read the features
This file is created by `src/build_features.py`.

In [2]:
DATA_DIR = "/opt/project/data/clean"

features = spark.read.parquet(f"{DATA_DIR}/features.parquet")
total_rows = features.count()

print(f"Rows: {total_rows:,}")
features.printSchema()

Rows: 769,950
root
 |-- service_date: date (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- target_outlier: boolean (nullable = true)
 |-- route_base: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- first_station_id: string (nullable = true)
 |-- last_station_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- next_station_id: string (nullable = true)
 |-- is_bus: integer (nullable = true)
 |-- stop_sequence: long (nullable = true)
 |-- departure_hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- scheduled_travel_min: double (nullable = true)
 |-- scheduled_dwell_min: double (nullable = true)
 |-- arrival_delay_min: double (nullable = true)
 |-- departure_delay_min: double (nullable = true)
 |-- target_next_arrival_delay_min: double (nullable = true)



### 4.3 Split into train and test

In [3]:
# The last day that belongs to train.
TRAIN_END = "2026-11-21"
is_train = F.col("service_date") <= F.to_date(F.lit(TRAIN_END))

train = features.filter(is_train)
test = features.filter(~is_train)

### 4.4 Save both parts
All models will use exactly the same data. Running this cell again replaces the old files.

In [4]:
train.write.mode("overwrite").parquet(f"{DATA_DIR}/train.parquet")
test.write.mode("overwrite").parquet(f"{DATA_DIR}/test.parquet")

# From now on, work with the saved files.
train = spark.read.parquet(f"{DATA_DIR}/train.parquet")
test = spark.read.parquet(f"{DATA_DIR}/test.parquet")

### 4.5 Check the split

In [5]:
train_rows = train.count()
test_rows = test.count()

print(f"Train: {train_rows:>8,}  ({train_rows / total_rows:.1%})")
print(f"Test:  {test_rows:>8,}  ({test_rows / total_rows:.1%})")

# No row may be lost or counted twice.
assert train_rows + test_rows == total_rows, "Some rows are lost or duplicated"

# All train dates must be earlier than all test dates.
train_last = train.agg(F.max("service_date")).first()[0]
test_first = test.agg(F.min("service_date")).first()[0]
assert train_last < test_first, "Train and test dates overlap"
print(f"Last train day: {train_last}, first test day: {test_first}")

print("\nRows per day:")
(train.withColumn("part", F.lit("train"))
 .unionByName(test.withColumn("part", F.lit("test")))
 .groupBy("service_date", "part").count()
 .orderBy("service_date")
 .show())

Train:  563,102  (73.1%)
Test:   206,848  (26.9%)


Last train day: 2026-11-21, first test day: 2026-11-22

Rows per day:


+------------+-----+------+
|service_date| part| count|
+------------+-----+------+
|  2026-11-17|train|115154|
|  2026-11-18|train|114692|
|  2026-11-19|train|115322|
|  2026-11-20|train|115389|
|  2026-11-21|train|102545|
|  2026-11-22| test| 91610|
|  2026-11-23| test|115238|
+------------+-----+------+



### 4.6 Fit and validation (parts of train)
They are not saved as files: they are easy to get from train by date.

In [6]:
# The day used to compare model variants.
VALIDATION_DAY = "2026-11-21"
validation_date = F.to_date(F.lit(VALIDATION_DAY))

fit_data = train.filter(F.col("service_date") < validation_date)
validation = train.filter(F.col("service_date") == validation_date)

fit_rows = fit_data.count()
validation_rows = validation.count()

print(f"Fit:        {fit_rows:>8,}")
print(f"Validation: {validation_rows:>8,}")

assert fit_rows + validation_rows == train_rows, "Fit + validation must equal train"

Fit:         460,557
Validation:  102,545


Step 4 is done. **Do not use `test` to choose or tune a model.**

While this notebook is connected, it can hold all Spark cores. Before running a script with
`spark-submit`, stop the notebook kernel (**Kernel → Shut Down Kernel**).